In [2]:
import os
from dotenv import load_dotenv, find_dotenv

# تحميل البيئة تلقائياً من ملف .env
load_dotenv(find_dotenv())

# التحقق من وجود مفتاح Groq
api_key = os.getenv("GROQ_API_KEY")
if api_key:
    print("✅ تم العثور على GROQ_API_KEY بنجاح!")
else:
    print("⚠️ لم يتم العثور على GROQ_API_KEY. يرجى التأكد من وجوده في ملف .env")


✅ تم العثور على GROQ_API_KEY بنجاح!


In [3]:
## Data Ingestion from website we need ti scrape the data
from langchain_community.document_loaders import  WebBaseLoader


/tmp/ipykernel_51310/829312346.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import  WebBaseLoader
/home/ali/ME/Code/AI/langchain-mastery/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
loader=WebBaseLoader("https://docs.smith.langchain.com/tutorials/Administrators/manage_spend")
loader

In [5]:
docs = loader.load()
print(docs)

[Document(metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content="LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics.LangSmith works wi

In [6]:
## chunk the document
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(docs)


In [7]:
chunks
print(f"Number of chunks: {len(chunks)}")
print(f"First chunk: {chunks[0].page_content[:500]}")  # Print the first 500 characters of the first chunk

Number of chunks: 5
First chunk: LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveRefere


In [8]:
import os
from dotenv import load_dotenv, find_dotenv

# البحث التلقائي عن ملف .env وتحميله
load_dotenv(find_dotenv())

# التحقق من وجود التوكين
hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
if hf_token and not hf_token.startswith("your_"):
    print("✅ تم تحميل Hugging Face Token وتفعيله بنجاح!")
else:
    print("ℹ️ يعمل النموذج الآن بدون توكين (النماذج العامة تعمل مجاناً وبدون مشاكل).")


✅ تم تحميل Hugging Face Token وتفعيله بنجاح!


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

hf_embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3694.79it/s]


In [10]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, hf_embeddings)

In [11]:
## smilarity search
query = "How to manage spend in LangChain?"
result = vectorstore.similarity_search(query , k=3)
print(f"Number of results: {len(result)}")
for i, res in enumerate(result):
    print(f"Result {i+1}: {res.page_content[:500]}")  # Print the first 500 characters of each result

Number of results: 3
Result 1: LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveRefere
Result 2: Copy the key and save it securely.Once your account and API key are ready, set up tracing:Set up tracingAdd tracing to your app in minutes with environment variables, framework integrations, or the SDK.Trace a RAG applicationFollow a step-by-step tutorial to instrument a retrieval-augmented generation app from start to finish.Investigate and monitorView tracesFilter, export, share, and compare traces via the UI or API.Monitor performanceBuild dashboards 

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant that answers questions based on the context provided.
    If the context does not contain the answer, respond with 'I don't know.'.

    Context:
    {context}

    Question: {question}
    Answer:"""
)

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7,
)

query = "what is LangSmith Observability"

document_chain = (
    {
        "context": lambda _: format_docs(result),
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

response = document_chain.invoke(query)

print(response)

LangSmith Observability is the part of LangSmith that gives you full visibility into your LLM‑powered applications. It lets you trace individual calls (including RAG pipelines), view and filter those traces, export or share them, and monitor production‑wide performance metrics. With built‑in dashboards, alerts, automations (rules, webhooks, evaluations) and the Engine for automatically detecting and diagnosing recurring issues, LangSmith Observability lets you monitor, debug, and improve the quality and reliability of your LLM applications.
